# Deteccion

In [25]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing import image

In [26]:
# Base model igual que en entrenamiento
base_model = applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(applications.resnet50.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

In [ ]:
import cv2
import time
import numpy as np
import tensorflow as tf
import os
from datetime import datetime

RUTA_MODELO = '~/clasificador.keras' # <-- CAMBIAR AQUÍ
class_names = ['Botellas_Plasticas', 'Latas_Metalicas', 'Mecato', 'Papel_Carton']
CARPETA_GUARDADO = "~/detecciones"

if not os.path.exists(CARPETA_GUARDADO):
    os.makedirs(CARPETA_GUARDADO)

print("Cargando el modelo de TerraLens...")
model.load_weights(RUTA_MODELO)
print("Modelo cargado correctamente.")

cap = cv2.VideoCapture(1)

def capturar_fondo():
    print("\n[SISTEMA] Calibrando fondo... No pongas nada frente a la cámara.")
    time.sleep(2)
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(gray, (21, 21), 0)

fondo_gray = capturar_fondo()
detectando = False
tiempo_inicio_deteccion = 0

print("[SISTEMA] Esperando residuos...")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    frame_mostrar = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    # Diferencia con el fondo
    diff = cv2.absdiff(fondo_gray, gray)
    _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    objeto_presente = any(cv2.contourArea(c) > 8000 for c in contornos)
    
    if objeto_presente:
        if not detectando:
            detectando = True
            tiempo_inicio_deteccion = time.time()
        
        tiempo_transcurrido = time.time() - tiempo_inicio_deteccion
        segundos_restantes = max(0, 3 - int(tiempo_transcurrido))
        
        cv2.putText(frame_mostrar, f"OBJETO DETECTADO - FOTO EN: {segundos_restantes}s", 
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        if tiempo_transcurrido >= 3:

            # PREDICCIÓN
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, (224, 224))
            img_array = np.expand_dims(img_resized, axis=0)
            
            pred = model.predict(img_array, verbose=0)[0]
            idx = np.argmax(pred)
            nombre_clase = class_names[idx]
            probabilidad = pred[idx] * 100
            
            # GUARDAR FOTO
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            nombre_archivo = f"{CARPETA_GUARDADO}/{nombre_clase}_{timestamp}.jpg"
            cv2.imwrite(nombre_archivo, frame)
            
            inicio_pausa = time.time()
            while time.time() - inicio_pausa < 3:
                res_frame = frame.copy()
                cv2.rectangle(res_frame, (10, 10), (600, 80), (0, 0, 0), -1)
                cv2.putText(res_frame, f"DETECTADO: {nombre_clase.upper()}", (20, 45), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                cv2.putText(res_frame, f"Confianza: {probabilidad:.1f}% | Guardado en /detecciones", (20, 70), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                cv2.imshow('TerraLens - Escaner Automotico', res_frame)
                cv2.waitKey(1)

            print(f"[LOG] Detectado: {nombre_clase} ({probabilidad:.1f}%) - Foto guardada.")
            
            # RESETEAR PARA EL SIGUIENTE OBJETO
            fondo_gray = capturar_fondo()
            detectando = False
    else:
        detectando = False
        cv2.putText(frame_mostrar, "ESTADO: ESPERANDO OBJETO...", 
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow('TerraLens - Escaner Automotico', frame_mostrar)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import time
import numpy as np
import tensorflow as tf
import os
from datetime import datetime

RUTA_MODELO = '~/clasificador.keras' # <-- CAMBIAR AQUÍ
class_names = ['Botellas_Plasticas', 'Latas_Metalicas', 'Mecato', 'Papel_Carton']
CARPETA_GUARDADO = "~/detecciones"

if not os.path.exists(CARPETA_GUARDADO):
    os.makedirs(CARPETA_GUARDADO)

print("Cargando el modelo de TerraLens...")
model.load_weights(RUTA_MODELO)
print("Modelo cargado correctamente.")

cap = cv2.VideoCapture(1)

def capturar_fondo():
    print("\n[SISTEMA] Calibrando fondo... No pongas nada frente a la cámara.")
    time.sleep(2)
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(gray, (21, 21), 0)

# Tomamos el fondo SOLO UNA VEZ al principio
fondo_gray = capturar_fondo()

estado = "ESPERANDO" 
tiempo_inicio_deteccion = 0
tiempo_inicio_resultado = 0
resultado_texto = ""
probabilidad_texto = ""

print("[SISTEMA] Listo. Esperando residuos...")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    frame_mostrar = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    # Diferencia con el fondo original
    diff = cv2.absdiff(fondo_gray, gray)
    _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Comprobar si hay algún objeto grande en pantalla
    objeto_presente = any(cv2.contourArea(c) > 5000 for c in contornos)
    
    if estado == "ESPERANDO":
        if objeto_presente:
            estado = "DETECTANDO"
            tiempo_inicio_deteccion = time.time()
        else:
            cv2.putText(frame_mostrar, "ESTADO: ESPERANDO OBJETO...", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.putText(frame_mostrar, "Presiona 'r' para recalibrar fondo", 
                        (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    elif estado == "DETECTANDO":
        if not objeto_presente:
            estado = "ESPERANDO"
        else:
            tiempo_transcurrido = time.time() - tiempo_inicio_deteccion
            segundos_restantes = max(0, 3 - int(tiempo_transcurrido))
            
            cv2.putText(frame_mostrar, f"OBJETO DETECTADO - FOTO EN: {segundos_restantes}s", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            if tiempo_transcurrido >= 3:
                # PREDICCIÓN
                img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, (224, 224))
                img_array = np.expand_dims(img_resized, axis=0)
                
                pred = model.predict(img_array, verbose=0)[0]
                idx = np.argmax(pred)
                nombre_clase = class_names[idx]
                probabilidad = pred[idx] * 100
                
                # GUARDAR FOTO
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                nombre_archivo = f"{CARPETA_GUARDADO}/{nombre_clase}_{timestamp}.jpg"
                cv2.imwrite(nombre_archivo, frame)
                print(f"[LOG] Detectado: {nombre_clase} ({probabilidad:.1f}%) - Foto guardada.")
                
                resultado_texto = f"DETECTADO: {nombre_clase.upper()}"
                probabilidad_texto = f"Confianza: {probabilidad:.1f}% | Guardado en carpeta"
                
                estado = "RESULTADO"
                tiempo_inicio_resultado = time.time()

    elif estado == "RESULTADO":
        # Mostrar el resultado congelado en pantalla por 3 segundos
        cv2.rectangle(frame_mostrar, (10, 10), (600, 80), (0, 0, 0), -1)
        cv2.putText(frame_mostrar, resultado_texto, (20, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(frame_mostrar, probabilidad_texto, (20, 70), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                    
        if time.time() - tiempo_inicio_resultado >= 3:
            estado = "ESPERANDO_RETIRO"

    elif estado == "ESPERANDO_RETIRO":
        if not objeto_presente:
            # Si el objeto ya NO está en pantalla, volvemos a escanear
            estado = "ESPERANDO"
        else:
            # Si el objeto sigue ahí, pedimos que lo quiten para no escanearlo doble
            cv2.rectangle(frame_mostrar, (10, 10), (600, 50), (0, 0, 0), -1)
            cv2.putText(frame_mostrar, "POR FAVOR, RETIRA EL OBJETO...", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

    cv2.imshow('TerraLens - Escaner Automatico', frame_mostrar)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):
        #recalibracion
        fondo_gray = capturar_fondo()
        estado = "ESPERANDO"

cap.release()
cv2.destroyAllWindows()